In [467]:
import struct
import random

_PRIVATE_RULE_SPEC_FMT = '>QQH?5x'
_PRIVATE_RULE_SPEC_SIZE = struct.calcsize(_PRIVATE_RULE_SPEC_FMT)

def pack_spec(mask_pre: int, mask_post: int, reps: int, flip: bool) -> bytes:
    return struct.pack(_PRIVATE_RULE_SPEC_FMT, mask_pre, mask_post, reps, flip)

def unpack_spec(packed: bytes) -> tuple[int, int, int, bool]:
    return struct.unpack(_PRIVATE_RULE_SPEC_FMT, packed)

# ---------- bit helpers ----------
def parity(x: int) -> int:              # faster than while-loop
    return bin(x).count('1') & 1

def extract_window_finite(state: int, i: int, L: int, N: int, pad_bit: int = 0) -> int:
    """Return the N-bit window that **starts** at absolute position `i`."""
    w = 0
    for j in range(N):
        idx = i + j
        bit = ((state >> idx) & 1) if 0 <= idx < L else pad_bit
        w = (w << 1) | bit
    return w

def apply_rule_expand(state: int, rule: int, L: int, N: int, pad_bits: int = 0) -> tuple[int, int]:
    grow   = N - 1
    new_L  = L + grow
    padded = (state << grow) | (pad_bits & ((1 << grow) - 1))

    out = 0
    for i in range(new_L):
        w     = extract_window_finite(padded, i, new_L, N)
        bit_i = (rule >> w) & 1
        out  |= bit_i << i
    return out, new_L

def implements_spec(pre: int, post: int, spec: bytes) -> bool:
    mask_pre, mask_post, _, flip = unpack_spec(spec)
    return parity(mask_pre & pre) ^ flip == parity(mask_post & post)

def implements_spec_or_opposite(pre: int, post: int, spec: bytes) -> bool:
    mask_pre, mask_post, _, _ = unpack_spec(spec)
    return (parity(mask_pre & pre) == parity(mask_post & post)
            or parity(mask_pre & pre) == ~parity(mask_post & post))


def random_bitmask(N: int, P: int) -> int:
    if N < 0 or P < 0:
        raise ValueError("N and P must be non-negative")
    if P > N:
        raise ValueError("P cannot exceed N")

    positions = random.sample(range(N), P)   # choose P distinct bit positions

    mask = 0
    for pos in positions:
        mask |= (1 << pos)

    return mask


def random_rule(N: int) -> int:
    return random.randrange(1 << N)


def get_random_spec1(N, M):
    L = N + M*(N - 1)
    mask_pre = random_bitmask(N, 1)
    mask_post = random_bitmask(L, 1)
    reps = M
    flip = random.choice([True, False])
    return pack_spec(mask_pre, mask_post, reps, flip)

def get_random_spec2(N, M):
    L = N + M*(N - 1)
    mask_pre = random_bitmask(N, 2)
    mask_post = random_bitmask(L, 2)
    reps = M
    flip = random.choice([True, False])
    return pack_spec(mask_pre, mask_post, reps, flip)

In [1089]:
N = 4
L_start = 22
M = 3
L = L_start + M*(N - 1)
print("L", L)
specs = [get_random_spec2(L_start, M)]

for spec in specs:
    mask_pre, mask_post, _, flip = unpack_spec(spec)

    print(f"mask pre :{mask_pre:0{N}b}")
    print(f"mask post:{mask_post:0{L}b}")
    print(f"     flip:{flip}")

rules = []

for rule in range(1 << (1 << N)):
    pad = (1 << N) - 1
    ok = True
    ok_flip = True

    for pre in range(1 << N):
        curr, curr_L = pre, L_start

        for _ in range(M):
            curr, curr_L = apply_rule_expand(curr, rule, curr_L, N, pad)

        for spec in specs:
            mask_pre, mask_post, reps, flip = unpack_spec(spec)
            flip_spec = pack_spec(mask_pre, mask_post, reps, not flip)

            if ok and not implements_spec(pre, curr, spec):
                ok = False

            if ok_flip and not implements_spec(pre, curr, flip_spec):
                ok_flip = False

            if not ok and not ok_flip:
                break

        if not ok and not ok_flip:
            break


    if ok or ok_flip:
        print(f"{rule:#0{2+(1<<N)//4}x}, reg: {ok}, flip: {ok_flip}")
        rules.append(rule)

# if len(rules):
    # print("FOUND: ", [f"{rule:#0{2+(1<<N)//4}x}" for rule in rules])


L 31
mask pre :10001000000
mask post:0000000000000000100000100000000
     flip:True
0x0000, reg: False, flip: True
0x0002, reg: False, flip: True
0x0004, reg: False, flip: True
0x0006, reg: False, flip: True
0x0008, reg: False, flip: True
0x000a, reg: False, flip: True
0x000c, reg: False, flip: True
0x000e, reg: False, flip: True
0x0014, reg: False, flip: True
0x0016, reg: False, flip: True
0x001c, reg: False, flip: True
0x001e, reg: False, flip: True
0x0020, reg: False, flip: True
0x0022, reg: False, flip: True
0x0024, reg: False, flip: True
0x0026, reg: False, flip: True
0x0028, reg: False, flip: True
0x002a, reg: False, flip: True
0x002c, reg: False, flip: True
0x002e, reg: False, flip: True
0x0034, reg: False, flip: True
0x0036, reg: False, flip: True
0x003c, reg: False, flip: True
0x003e, reg: False, flip: True
0x0040, reg: False, flip: True
0x0042, reg: False, flip: True
0x0044, reg: False, flip: True
0x0046, reg: False, flip: True
0x0048, reg: False, flip: True
0x004a, reg: Fals

KeyboardInterrupt: 

In [93]:
def extract_window_right_pad(state: int, i: int, L: int, N: int, pad_bits: int) -> int:
    w = 0

    for j in range(N):
        idx = i + j

        if idx < L:
            bit = (state >> idx) & 1
        else:
            bit = (pad_bits >> (idx - L)) & 1

        w = (w << 1) | bit
    return w


def apply_rule_append_pad(state: int,
                          rule: int,
                          L: int,
                          N: int,
                          pad_bits: int) -> tuple[int, int]:
    grow  = N - 1
    new_L = L + grow
    out   = 0

    state = state << grow
    grow_mask = (1 << grow) - 1
    state |= pad_bits & grow_mask

    for i in range(L):
        w     = extract_window_finite(state, i, L, N)
        bit_i = (rule >> w) & 1
        out  |= bit_i << i

    return out, new_L

# Rule-30, N = 4 → grow = 3 (window length 4, right-expanding)
rule30   = 0b10111110
state, L = 0b1, 1          # initial single '1'
N        = 3
pad_bits = 0b01           # bits “101” will be glued onto the right each step

for t in range(5):
    state, L = apply_rule_append_pad(state, rule30, L, N, pad_bits)
    print(f"{state:0{L}b}")


001
00101
0010101
001010101
00101010101
